# Exp 1: Chinese text segmentation

In [1]:
import itertools as it
import json
import os
import re
import typing as ty

import numpy as np
import pandas as pd

TRAIN_CORPUS_PATH = "dataset/trainCorpus.txt"
STOP_WORDS_PATH = "dataset/stopword.txt"
TEST_CORPUS_PATH = "dataset/flightnews.txt"

try:
    os.mkdir("out/")
except:
    pass

## 1. SBME tagging

Chinese sentences are written without explicit separator. Zhao et al.[^0] proposed a tagging scheme that maps to a hidden Markov model for recognizing word boundaries.

* S: A single-character word.
* B: Beginning of a word.
* M: Middle of a word.
* E: End of a word.

Example:

```plain-text
深圳 地铁 将 设立
BE BE S BE
```

By marking the characters as emission and `{S,B,M,E}` as hidden states, we can build a hMM and try to deduce its SBME sequence from the characters.

[^0]: https://aclanthology.org/Y06-1012/

### 1.1. Tag SBME

In [2]:
def isplit(x: str, sep: str | None = None) -> ty.Generator[str]:
    for match in re.finditer(r"\S+", x) if sep is None else re.finditer(re.escape(sep), x):
        yield x[match.start() : match.end()]


In [3]:
SBMETag = ty.Literal["S", "B", "M", "E"]
TAGS: list[SBMETag] = ["S", "B", "M", "E"]

class TaggedChar(ty.NamedTuple):
    char: str
    tag: SBMETag

    def __str__(self) -> str:
        return f"{self.char}/{self.tag}"


def tag_sbme(x: ty.Generator[str]) -> ty.Generator[TaggedChar]:
    for word in x:
        if len(word) == 1:
            yield TaggedChar(word, "S")
        else:
            yield TaggedChar(word[0], "B")
            for ch in word[1:-1]:
                yield TaggedChar(ch, "M")
            yield TaggedChar(word[-1], "E")

In [4]:
with open(TRAIN_CORPUS_PATH) as f:
    train_corpus = f.read()

print(" ".join((str(v) for v in it.islice(tag_sbme(isplit(train_corpus)), 10))), "...")

深/B 圳/E 地/B 铁/E 将/S 设/B 立/E Ｖ/S Ｉ/S Ｐ/S ...


### 1.2. The Viterbi algorithm

Pseudocode from Wikipedia[^1] is already a good reference.

[^1]: https://en.wikipedia.org/wiki/Viterbi_algorithm

In [5]:
TState = ty.TypeVar("TState")
TEmission = ty.TypeVar("TEmission")


class Viterbi(ty.Generic[TState, TEmission], ty.NamedTuple):
    states: list[TState]
    emission_map: dict[TEmission, int]
    init: np.ndarray
    trans: np.ndarray
    emit: np.ndarray
    unk_idx: int = 0

    @property
    def n_states(self) -> int:
        return len(self.states)

    def __call__(self, obs: ty.Iterable[TEmission]) -> list[TState]:
        obs_indices = [self.emission_map.get(o, self.unk_idx) for o in obs]

        S = self.n_states
        T = len(obs_indices)

        prob = np.zeros((T, S))
        prev = np.zeros((T, S), dtype=np.int64)

        for s in range(S):
            prob[0, s] = self.init[s] * self.emit[s, obs_indices[0]]

        for t in range(1, T):
            for s in range(S):
                for r in range(S):
                    new_prob = (
                        prob[t - 1, r] * self.trans[r, s] * self.emit[s, obs_indices[t]]
                    )
                    if new_prob > prob[t, s]:
                        prob[t, s] = new_prob
                        prev[t, s] = r

        path: list[TState | None] = [None] * T
        path[T - 1] = self.states[int(np.argmax(prob[T - 1, :]))]
        for t in range(T - 2, -1, -1):
            tp1_index = self.states.index(path[t + 1])  # type: ignore[union-attr]
            path[t] = self.states[int(prev[t + 1, tp1_index])]

        assert all(x is not None for x in path)
        return path  # type: ignore[list-item]

## 2. Parameter estimation

We collect the defined states and emissions, and observed transition/emission matrices over the training corpus.

In [6]:
SBME_INDICES: dict[SBMETag, int] = {"S": 0, "B": 1, "M": 2, "E": 3}

In [7]:
def estimate(corpus_path: str, lam: float = 1.0) -> Viterbi[SBMETag, str]:
    with open(corpus_path, encoding="utf-8") as f:
        lines = [line for line in f.readlines() if line.strip()]

    chars: set[str] = set()
    for line in lines:
        for word in isplit(line):
            chars.update(word)
    char_to_idx: dict[str, int] = {"<UNK>": 0}
    for ch in sorted(chars):
        char_to_idx[ch] = len(char_to_idx)

    S = len(TAGS)
    V = len(char_to_idx)

    # Max likelihood estimation
    init_cnt = np.zeros(S, dtype=np.float64)
    trans_cnt = np.zeros((S, S), dtype=np.float64)
    emit_cnt = np.zeros((S, V), dtype=np.float64)
    n_sentences = 0

    for line in lines:
        n_sentences += 1
        tag_ids: list[int] = []
        char_ids: list[int] = []
        for tc in tag_sbme(isplit(line)):
            tag_ids.append(SBME_INDICES[tc.tag])
            char_ids.append(char_to_idx[tc.char])
        init_cnt[tag_ids[0]] += 1.0
        for i in range(len(tag_ids) - 1):
            trans_cnt[tag_ids[i], tag_ids[i + 1]] += 1.0
        for i in range(len(tag_ids)):
            emit_cnt[tag_ids[i], char_ids[i]] += 1.0

    # Normalize with Laplace smoothing
    init = (init_cnt + lam) / (n_sentences + lam * S)
    trans = (trans_cnt + lam) / (trans_cnt.sum(axis=1, keepdims=True) + lam * S)
    emit = (emit_cnt + lam) / (emit_cnt.sum(axis=1, keepdims=True) + lam * V)

    return Viterbi(
        states=TAGS,
        emission_map=char_to_idx,
        init=init,
        trans=trans,
        emit=emit,
        unk_idx=char_to_idx["<UNK>"],
    )

In [8]:
v = estimate(TRAIN_CORPUS_PATH)

print(f"vocab size:\t{v.emit.shape[1]}")
print(f"init shape:\t{v.init.shape}")
print(f"trans shape:\t{v.trans.shape}")
print(f"emit shape:\t{v.emit.shape}")
print()

print("pi:")
print(pd.Series(v.init, index=v.states))
print()

print("A:")
print(pd.DataFrame(v.trans, index=v.states, columns=v.states))
print()

vocab size:	2576
init shape:	(4,)
trans shape:	(4, 4)
emit shape:	(4, 2576)

pi:
S    0.356351
B    0.643292
M    0.000179
E    0.000179
dtype: float64

A:
          S         B         M         E
S  0.449173  0.550728  0.000049  0.000049
B  0.000032  0.000032  0.152227  0.847709
M  0.000149  0.000149  0.292392  0.707310
E  0.387899  0.612026  0.000037  0.000037



### 2.3. Save Probability Tables

In [9]:
np.save("out/init.npy", v.init)
np.save("out/trans.npy", v.trans)
np.save("out/emit.npy", v.emit)

with open("out/init.json", "w", encoding="utf-8") as f:
    json.dump(
        {t: float(v.init[i]) for i, t in enumerate(v.states)},
        f,
        ensure_ascii=False,
    )

with open("out/transition_probability.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            from_t: {to_t: float(v.trans[i, j]) for j, to_t in enumerate(v.states)}
            for i, from_t in enumerate(v.states)
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

idx_to_char = {idx: ch for ch, idx in v.emission_map.items()}
emit_dict: dict[str, dict[str, float]] = {}
for i, tag in enumerate(v.states):
    emit_dict[tag] = {}
    for j in range(v.emit.shape[1]):
        ch = idx_to_char[j]
        prob = float(v.emit[i, j])
        if prob > 1e-8:
            emit_dict[tag][ch] = prob

with open("out/emit_dict.json", "w", encoding="utf-8") as f:
    json.dump(emit_dict, f, ensure_ascii=False, indent=2)

with open("out/char_to_idx.json", "w", encoding="utf-8") as f:
    json.dump(v.emission_map, f, ensure_ascii=False, indent=2, sort_keys=True)

## 3. Evaluation

### 3.1. The cut function

In [10]:
def cut(sentence: str, model: Viterbi[SBMETag, str]) -> ty.Generator[str]:
    tags = model(list(sentence))

    buf: list[str] = []
    for ch, tag in zip(sentence, tags):
        if tag == "S":
            yield ch
        elif tag == "B":
            buf = [ch]
        elif tag == "M":
            buf.append(ch)
        elif tag == "E":
            buf.append(ch)
            yield "".join(buf)
            buf = []

    if buf:
        yield "".join(buf)

In [11]:
SAMPLE = "深航科技攀枝花机场遇险：机腹轮胎均疑受损，跑道灯部分损坏"
words = cut(SAMPLE, v)

print(SAMPLE)
print("->")
print(" | ".join(words))

深航科技攀枝花机场遇险：机腹轮胎均疑受损，跑道灯部分损坏
->
深航 | 科技 | 攀枝 | 花机场 | 遇险 | ： | 机腹 | 轮胎均 | 疑受 | 损，跑 | 道灯 | 部分 | 损坏


### 3.2. Stop words

In [12]:
with open(STOP_WORDS_PATH, encoding="utf-8") as f:
    stop_words = {line.strip() for line in f if line.strip()}

### 3.2. Segment Test Corpus and Extract Top-10 Frequent Words

In [13]:
from collections import Counter

with open(TEST_CORPUS_PATH, encoding="utf-8") as f:
    test_lines = [line.strip() for line in f if line.strip()]

all_words: list[str] = []
segmented_lines: list[str] = []
with open("out/segmented_flightnews.txt", "w", encoding="utf-8") as f:
    for line in test_lines:
        words = cut(line, v)
        all_words.extend(words)
        f.write("\t".join(words))
        f.write("\n")

print(f"Total words before stop word filtering: {len(all_words)}")
filtered = [w for w in all_words if w not in stop_words]
print(f"Total words after stop word filtering: {len(filtered)}")

freq = Counter(filtered)
top10 = freq.most_common(10)

print("\nTop-10 Most Frequent Words (HMM Segmentation)")
print(
    pd.DataFrame(top10, columns=["Word", "Count"]).set_index(
        pd.Index(range(1, len(top10) + 1), name="Rank")
    )
)

Total words before stop word filtering: 1320
Total words after stop word filtering: 989

Top-10 Most Frequent Words (HMM Segmentation)
     Word  Count
Rank            
1       机     26
2       跑     14
3      报告     13
4       后     13
5      飞机     11
6       飞     11
7      新闻     10
8       航     10
9       场     10
10     上游      9


### 3.3. Comparison with Jieba

In [14]:
import jieba

jieba_words: list[str] = []
for line in test_lines:
    jieba_words.extend(jieba.lcut(line))

jieba_filtered = [w for w in jieba_words if w not in stop_words]
jieba_freq = Counter(jieba_filtered)
jieba_top10 = jieba_freq.most_common(10)

print("\nTop-10 Most Frequent Words (jieba)")
print(
    pd.DataFrame(jieba_top10, columns=["Word", "Count"]).set_index(
        pd.Index(range(1, len(jieba_top10) + 1), name="Rank")
    )
)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\MANTLE~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.413 seconds.
Prefix dict has been built successfully.



Top-10 Most Frequent Words (jieba)
        Word  Count
Rank               
1         跑道     20
2         飞机     19
3         航班     17
4        攀枝花     16
5         机场     15
6          后     12
7         落地     12
8         检查     10
9     ZH9247     10
10        机长      9
